# 1) Load CSVs and Build Unit Records
Read the three input CSVs (first 6 orders). Each row in the CSVs corresponds to one order, and columns are item entries. Build one unit record per physical item, then summarize order sizes.

In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
from copy import deepcopy
from math import ceil

NUM_ORDERS = 6

itemtypes = pd.read_csv("order_itemtypes.csv", header=None, nrows=NUM_ORDERS)
quantities = pd.read_csv("order_quantities.csv", header=None, nrows=NUM_ORDERS)
totes_csv = pd.read_csv("orders_totes.csv", header=None, nrows=NUM_ORDERS)

# Build per-order demand and per-tote inventory, plus individual unit records
order_demand = defaultdict(lambda: defaultdict(int))
tote_inventory = defaultdict(lambda: defaultdict(int))
units = []

for o in range(itemtypes.shape[0]):
    for k in range(itemtypes.shape[1]):
        it = itemtypes.iat[o, k]
        qt = quantities.iat[o, k] if k < quantities.shape[1] else pd.NA
        tt = totes_csv.iat[o, k] if k < totes_csv.shape[1] else pd.NA
        if pd.notna(it) and pd.notna(qt) and pd.notna(tt):
            it, qt, tt = int(it), int(qt), int(tt)
            order_demand[o + 1][it] += qt
            tote_inventory[tt][it] += qt
            for _ in range(qt):
                units.append({"order": o + 1, "item_type": it, "tote": tt})

units_df = pd.DataFrame(units)
orders = sorted(order_demand.keys())
tote_ids = sorted(tote_inventory.keys())

order_sizes = {o: sum(order_demand[o].values()) for o in orders}

print("Order demand (item_type: qty):")
for o in orders:
    parts = [f"item {it} x {q}" for it, q in sorted(order_demand[o].items())]
    print(f"  Order {o} ({order_sizes[o]} units): {', '.join(parts)}")

print(f"\nTote inventory:")
for t in tote_ids:
    parts = [f"item {it} x {q}" for it, q in sorted(tote_inventory[t].items())]
    print(f"  Tote {t}: {', '.join(parts)}")

print(f"\nTotal units: {len(units_df)}")
print(f"Total orders: {len(orders)}")
print(f"Total totes: {len(tote_ids)}")

Order demand (item_type: qty):
  Order 1 (5 units): item 1 x 2, item 3 x 3
  Order 2 (5 units): item 2 x 1, item 3 x 3, item 4 x 1
  Order 3 (2 units): item 5 x 2
  Order 4 (4 units): item 0 x 3, item 5 x 1
  Order 5 (2 units): item 1 x 1, item 2 x 1
  Order 6 (2 units): item 1 x 2

Tote inventory:
  Tote 0: item 1 x 2, item 3 x 3
  Tote 1: item 1 x 2
  Tote 4: item 3 x 3
  Tote 7: item 5 x 2
  Tote 9: item 0 x 3
  Tote 10: item 5 x 1
  Tote 11: item 1 x 1
  Tote 12: item 2 x 1
  Tote 14: item 2 x 1, item 4 x 1

Total units: 20
Total orders: 6
Total totes: 9


# 2) Greedy Algorithm: SPT + Earliest-Free Conveyor + Urgency Tote Sequencing
**Phase 1** — Sort orders by total units ascending (SPT).
**Phase 2** — Assign each order to the least-loaded conveyor.
**Phase 3** — Build tote release sequence greedily: at each step, release the tote that contributes the most to currently-active orders (weighted by urgency).

In [2]:
NUM_CONVEYORS = 4

# --- Phase 1: SPT ordering ---
spt_order = sorted(orders, key=lambda o: (order_sizes[o], o))
print("SPT order (by total units):")
for o in spt_order:
    print(f"  Order {o}: {order_sizes[o]} units")

# --- Phase 2: Conveyor assignment (least-loaded machine) ---
conveyor_load = {c: 0 for c in range(1, NUM_CONVEYORS + 1)}
order_to_conveyor = {}
conveyor_queues = defaultdict(list)

for o in spt_order:
    c_star = min(conveyor_load, key=conveyor_load.get)
    order_to_conveyor[o] = c_star
    conveyor_queues[c_star].append(o)
    conveyor_load[c_star] += order_sizes[o]

print("\nConveyor assignment:")
for c in sorted(conveyor_queues):
    queue_str = " -> ".join(str(o) for o in conveyor_queues[c])
    print(f"  Conveyor {c}: [{queue_str}] (load={conveyor_load[c]} units)")

# --- Phase 3: Urgency-based tote sequencing ---
remaining = deepcopy(dict(order_demand))
available_totes = deepcopy(dict(tote_inventory))
queue_pos = {}
for c, queue in conveyor_queues.items():
    for pos, o in enumerate(queue):
        queue_pos[o] = pos

finished_orders = set()
tote_sequence = []

def get_active_orders():
    """Return the set of currently-active orders (first unfinished per conveyor)."""
    active = {}
    for c, queue in conveyor_queues.items():
        for o in queue:
            if o not in finished_orders:
                active[c] = o
                break
    return active

def is_order_done(o):
    return all(q <= 0 for q in remaining[o].values())

while len(finished_orders) < len(orders):
    active = get_active_orders()
    if not active:
        break

    active_set = set(active.values())

    best_tote = None
    best_score = -1

    for t, inv in available_totes.items():
        score = 0.0
        for o in active_set:
            urgency = 1.0 / (1 + queue_pos[o])
            for it, q_avail in inv.items():
                if q_avail > 0 and it in remaining[o] and remaining[o][it] > 0:
                    score += min(q_avail, remaining[o][it]) * urgency
        if score > best_score:
            best_score = score
            best_tote = t

    if best_tote is None or best_score <= 0:
        # No tote can help any active order — release any remaining tote
        if available_totes:
            best_tote = next(iter(available_totes))
        else:
            break

    tote_sequence.append(best_tote)

    # Deduct fulfilled items from ALL orders (not just active), since the
    # tote's items are physically released and can be picked by any order.
    inv = available_totes[best_tote]
    for o in orders:
        if o in finished_orders:
            continue
        for it in list(inv.keys()):
            if inv[it] > 0 and it in remaining[o] and remaining[o][it] > 0:
                used = min(inv[it], remaining[o][it])
                remaining[o][it] -= used
                inv[it] -= used

    # Remove empty tote
    if all(q <= 0 for q in inv.values()):
        del available_totes[best_tote]

    # Check for newly finished orders
    for o in orders:
        if o not in finished_orders and is_order_done(o):
            finished_orders.add(o)

print("\nTote release sequence:")
print(f"  {tote_sequence}")

print("\nOrder completion status:")
for o in orders:
    status = "DONE" if o in finished_orders else f"remaining: {dict(remaining[o])}"
    print(f"  Order {o}: {status}")

SPT order (by total units):
  Order 3: 2 units
  Order 5: 2 units
  Order 6: 2 units
  Order 4: 4 units
  Order 1: 5 units
  Order 2: 5 units

Conveyor assignment:
  Conveyor 1: [3 -> 1] (load=7 units)
  Conveyor 2: [5 -> 2] (load=7 units)
  Conveyor 3: [6] (load=2 units)
  Conveyor 4: [4] (load=4 units)

Tote release sequence:
  [0, 7, 9, 1, 14, 10, 11, 12, 4]

Order completion status:
  Order 1: DONE
  Order 2: DONE
  Order 3: DONE
  Order 4: DONE
  Order 5: DONE
  Order 6: DONE


# 3) Simulate Pick Times
Given the tote release sequence and conveyor assignments, assign release slots to units and compute recirculation-aware pick times. Parameters from the README: `slot_time_sec=1`, `belt_time_sec=2`.

In [3]:
SLOT_TIME = 1   # seconds per release slot
BELT_TIME = 2   # seconds per conveyor traversal

# Build the release slot assignment from the tote sequence.
# Units from each tote are released in contiguous slots, in the order
# they appear in units_df (preserving original CSV row order within a tote).
slot_assignments = []
slot = 0

for t in tote_sequence:
    tote_units = units_df[units_df["tote"] == t].copy()
    for _, u in tote_units.iterrows():
        slot_assignments.append({
            "slot": slot,
            "order": u["order"],
            "item_type": u["item_type"],
            "tote": u["tote"],
            "conveyor": order_to_conveyor[u["order"]],
        })
        slot += 1

schedule = pd.DataFrame(slot_assignments)

# Compute pick times with recirculation.
# pick_time = slot*SLOT_TIME + BELT_TIME + (c-1)*BELT_TIME + BELT_TIME/2 + 4*BELT_TIME*k
# where k is the minimum non-negative integer such that pick_time >= conveyor_ready[c]
# (per-conveyor one-order-at-a-time: a conveyor can only pick one order at a time,
#  and within an order, items are picked sequentially as they arrive)

conveyor_ready = {c: 0.0 for c in range(1, NUM_CONVEYORS + 1)}
# Track which order is currently being picked on each conveyor
conveyor_current_order = {c: None for c in range(1, NUM_CONVEYORS + 1)}

pick_times = []
recirc_loops = []

for _, row in schedule.iterrows():
    s = row["slot"]
    c = row["conveyor"]
    o = row["order"]

    first_arrival = s * SLOT_TIME + BELT_TIME + (c - 1) * BELT_TIME + BELT_TIME / 2.0
    loop_period = 4 * BELT_TIME

    # The conveyor must be free for this order. If a different order is active
    # on this conveyor, items for this order recirculate until the conveyor finishes.
    ready = conveyor_ready[c]

    if first_arrival >= ready:
        pick_time = first_arrival
        k = 0
    else:
        k = ceil((ready - first_arrival) / loop_period)
        pick_time = first_arrival + k * loop_period

    pick_times.append(pick_time)
    recirc_loops.append(k)
    conveyor_ready[c] = pick_time
    conveyor_current_order[c] = o

schedule["pick_time"] = pick_times
schedule["recirc_loops"] = recirc_loops
schedule["release_time"] = schedule["slot"] * SLOT_TIME

# Compute order completion times
order_completion = schedule.groupby("order")["pick_time"].max()
order_start = schedule.groupby("order")["pick_time"].min()
objective = order_completion.sum()

print("Release schedule with pick times:")
print(schedule[["slot", "release_time", "pick_time", "order", "item_type", "tote", "conveyor", "recirc_loops"]].to_string(index=False))

print(f"\nOrder timing:")
for o in spt_order:
    c = order_to_conveyor[o]
    print(f"  Order {o} (conv {c}): start={order_start[o]:.1f}s, completion={order_completion[o]:.1f}s")

print(f"\nObjective (sum of completion times): {objective:.1f}s")

Release schedule with pick times:
 slot  release_time  pick_time  order  item_type  tote  conveyor  recirc_loops
    0             0        3.0      1          3     0         1             0
    1             1        4.0      1          3     0         1             0
    2             2        5.0      1          3     0         1             0
    3             3        6.0      1          1     0         1             0
    4             4        7.0      1          1     0         1             0
    5             5        8.0      3          5     7         1             0
    6             6        9.0      3          5     7         1             0
    7             7       16.0      4          0     9         4             0
    8             8       17.0      4          0     9         4             0
    9             9       18.0      4          0     9         4             0
   10            10       17.0      6          1     1         3             0
   11            1

# 4) Generate Conveyor Input CSV
Produce `greedy_heuristic_output.csv` in the conveyor-input format, with orders processed in the greedy-optimized sequence and conveyor assignments. One row per order.

In [4]:
ITEM_COLS = ["cirle", "pentagon", "trapezoid", "triangle", "star", "moon", "heart", "cross"]

order_start_times = schedule.groupby("order")["pick_time"].min().to_dict()
output_order = sorted(spt_order, key=lambda o: order_start_times.get(o, float("inf")))

csv_rows = []
for o in output_order:
    conv = order_to_conveyor[o]
    row = {"conv_num": conv}
    for it in range(8):
        row[it] = order_demand[o].get(it, 0)
    csv_rows.append(row)

out = pd.DataFrame(csv_rows)
out = out[["conv_num"] + list(range(8))]
out.columns = ["conv_num"] + ITEM_COLS

output_file = "greedy_heuristic_output.csv"
out.to_csv(output_file, index=False)

print(f"Generated: {output_file}")
print(out.to_string(index=False))

print(f"\nOrder -> Conveyor (greedy sequence):")
for o in output_order:
    print(f"  Order {o} -> Conveyor {order_to_conveyor[o]}")

print(f"\nGreedy objective (sum completion times): {objective:.1f}s")

Generated: greedy_heuristic_output.csv
 conv_num  cirle  pentagon  trapezoid  triangle  star  moon  heart  cross
        1      0         2          0         3     0     0      0      0
        1      0         0          0         0     0     2      0      0
        4      3         0          0         0     0     1      0      0
        3      0         2          0         0     0     0      0      0
        2      0         0          1         3     1     0      0      0
        2      0         1          1         0     0     0      0      0

Order -> Conveyor (greedy sequence):
  Order 1 -> Conveyor 1
  Order 3 -> Conveyor 1
  Order 4 -> Conveyor 4
  Order 6 -> Conveyor 3
  Order 2 -> Conveyor 2
  Order 5 -> Conveyor 2

Greedy objective (sum completion times): 102.0s
